# HeritageAR YOLOv8 Training - Google Colab
## Function 1: Sigiriya Sub-Landmark Detection

This notebook trains YOLOv8 for Sigiriya landmark detection in Google Colab using a Roboflow dataset ZIP file.

Quick workflow:
1. Mount Google Drive (for backup)
2. Upload Roboflow YOLOv8 ZIP dataset
3. Extract and verify structure
4. Create data.yaml config
5. Train YOLOv8n model
6. Export to TensorFlow Lite
7. Download trained model for Flutter

No GitHub clone or local dependencies needed — everything runs in Colab!


## Step 1: Mount Google Drive
Save trained models and results to your Drive for backup and download.


## Step 1.5: Install Training Dependencies
Install the Python package needed for YOLOv8 training in Colab.


In [ ]:
!pip install -U ultralytics opencv-python pyyaml
print('✓ ultralytics installed successfully!')


In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive so you can save checkpoints and downloaded models
# to persistent storage.
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")


## Step 2: Prepare Dataset Folder
Set up the folder structure for the Roboflow YOLOv8 dataset.


In [ ]:
import os
from pathlib import Path

# The dataset should be extracted to /content/sigiriya_dataset.yolov8 or /content/sigiriya_dataset
# Expected structure after extraction:
# /content/<dataset_root>/train/images
# /content/<dataset_root>/train/labels
# /content/<dataset_root>/valid/images
# /content/<dataset_root>/valid/labels
# /content/<dataset_root>/test/images
# /content/<dataset_root>/test/labels
dataset_path = Path("/content/sigiriya_dataset")
dataset_path.mkdir(parents=True, exist_ok=True)

print("Dataset folder setup:", dataset_path)
print("\nYour classes:")
classes = [
    "sigiriya_lion_paws",
    "sigiriya_lion_rock",
    "sigiriya_mirror_wall",
    "sigiriya_throne",
    "sigiriya_ticket_counter"
]
for idx, cls in enumerate(classes):
    print(f"  {idx}: {cls}")


## Step 3: Upload and Extract Roboflow Dataset ZIP
⚠️ **IMPORTANT:**
1. Download your annotated dataset from Roboflow as "YOLOv8" format (`.zip`)
2. Run the cell below
3. Upload the `.zip` file when prompted
4. Wait for extraction to complete


In [ ]:
from google.colab import files
import zipfile

print("Uploading Roboflow dataset ZIP file...")
print("Click 'Choose Files' and select your exported YOLOv8 dataset")

uploaded = files.upload()

# Get the uploaded zip file
zip_file = list(uploaded.keys())[0]
print(f"✓ Uploaded: {zip_file}")

# Extract to sigiriya_dataset
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("✓ Dataset extracted!")

# List contents
!ls -la /content/sigiriya_dataset/

## Step 4: Create data.yaml
Tell YOLOv8 where to find the training/validation/test data.


In [ ]:
import yaml
from pathlib import Path

def find_dataset_root():
    # Search common locations first, then fall back to a recursive scan under /content
    candidate_names = [
        'sigiriya_dataset',
        'sigiriya_dataset.yolov8',
        'sigiriya_dataset.v3-sigiriya_dataset_80-10-10.yolov8',
        'dataset',
    ]

    for name in candidate_names:
        direct = Path('/content') / name
        if (direct / 'train').exists() or (direct / 'images').exists() or (direct / 'data.yaml').exists():
            return direct

    for match in Path('/content').rglob('data.yaml'):
        parent = match.parent
        if (parent / 'train').exists() or (parent / 'images').exists() or (parent / 'valid').exists() or (parent / 'test').exists():
            return parent

    for folder in Path('/content').rglob('*'):
        if not folder.is_dir():
            continue
        if (folder / 'train').exists() or (folder / 'images').exists() or (folder / 'valid').exists() or (folder / 'test').exists():
            return folder

    return None

dataset_root = find_dataset_root()

if dataset_root is None:
    raise FileNotFoundError('Could not find the extracted Roboflow dataset folder under /content. Run !ls -la /content to inspect the uploaded ZIP location.')

# Create data.yaml for YOLOv8 with your classes
# Roboflow exports usually store split folders at the dataset root
data_yaml_content = {
    'path': str(dataset_root),
    'train': 'train/images',
    'val': 'valid/images',  # Roboflow uses 'valid' not 'val'
    'test': 'test/images',
    'nc': 5,  # number of classes
    'names': {
        0: 'sigiriya_lion_paws',
        1: 'sigiriya_lion_rock',
        2: 'sigiriya_mirror_wall',
        3: 'sigiriya_throne',
        4: 'sigiriya_ticket_counter'
    }
}

yaml_path = dataset_root / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print('✓ data.yaml created at:', yaml_path)
print('\nContents:')
with open(yaml_path, 'r') as f:
    print(f.read())


## Step 5: Verify Dataset Structure
Check that all image and label files are extracted correctly.


In [ ]:
# Verify dataset structure
import os
from pathlib import Path

def find_dataset_root():
    candidate_names = [
        'sigiriya_dataset',
        'sigiriya_dataset.yolov8',
        'sigiriya_dataset.v3-sigiriya_dataset_80-10-10.yolov8',
        'dataset',
    ]

    for name in candidate_names:
        direct = Path('/content') / name
        if (direct / 'train').exists() or (direct / 'images').exists() or (direct / 'data.yaml').exists():
            return direct

    for match in Path('/content').rglob('data.yaml'):
        parent = match.parent
        if (parent / 'train').exists() or (parent / 'images').exists() or (parent / 'valid').exists() or (parent / 'test').exists():
            return parent

    for folder in Path('/content').rglob('*'):
        if not folder.is_dir():
            continue
        if (folder / 'train').exists() or (folder / 'images').exists() or (folder / 'valid').exists() or (folder / 'test').exists():
            return folder

    return None

dataset_path = find_dataset_root()

if dataset_path is None:
    print('Dataset folder not found in expected locations.')
    print('Check /content and use the exact extracted folder name.')
    !ls -la /content
else:
    print('Dataset Structure:')
    print('='*50)

    # Check images and labels in common Roboflow layouts
    split_map = {
        'train': ['train', 'train/images', 'images/train'],
        'val': ['valid', 'val', 'valid/images', 'val/images', 'images/valid', 'images/val'],
        'test': ['test', 'test/images', 'images/test'],
    }

    for split, options in split_map.items():
        img_path = None
        label_path = None

        for option in options:
            candidate = dataset_path / option
            if candidate.exists():
                # Prefer the image directory if this is a split folder
                if candidate.is_dir() and (candidate / 'images').exists():
                    img_path = candidate / 'images'
                else:
                    img_path = candidate
                break

        # try matching labels folder near the image folder
        if img_path is not None:
            # Better detection: look beside the images folder based on actual path
            if img_path.name == 'images' and (img_path.parent / 'labels').exists():
                label_path = img_path.parent / 'labels'
            elif img_path.parent.name == 'images' and (img_path.parent.parent / 'labels' / img_path.name).exists():
                label_path = img_path.parent.parent / 'labels' / img_path.name
            else:
                # common Roboflow structures fallback
                if (dataset_path / split / 'labels').exists():
                    label_path = dataset_path / split / 'labels'
                elif (dataset_path / 'labels' / split).exists():
                    label_path = dataset_path / 'labels' / split
                elif (dataset_path / 'valid' / 'labels').exists() and split == 'val':
                    label_path = dataset_path / 'valid' / 'labels'

            img_count = len([p for p in img_path.rglob('*') if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])
            label_count = len([p for p in label_path.iterdir() if p.is_file()]) if label_path and label_path.exists() else 0
            print(f'✓ {split.upper()}: {img_count} images, {label_count} labels')
        else:
            print(f'✗ {split.upper()}: Missing folder')

    print('\n✓ Dataset verification complete!')

## Step 6: Train YOLOv8 Model with Moderate Augmentation
Use YOLOv8's built-in training augmentation. Since Roboflow preprocessing/split is already done, keep this moderate and realistic for heritage structures.


## Step 6: Train YOLOv8 Model
Start training on the dataset. This may take 30-60 minutes depending on GPU.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import os

def find_dataset_root():
    candidate_names = [
        'sigiriya_dataset',
        'sigiriya_dataset.yolov8',
        'sigiriya_dataset.v3-sigiriya_dataset_80-10-10.yolov8',
        'dataset',
    ]

    for name in candidate_names:
        direct = Path('/content') / name
        if (direct / 'train').exists() or (direct / 'images').exists() or (direct / 'data.yaml').exists():
            return direct

    for match in Path('/content').rglob('data.yaml'):
        parent = match.parent
        if (parent / 'train').exists() or (parent / 'images').exists() or (parent / 'valid').exists() or (parent / 'test').exists():
            return parent

    for folder in Path('/content').rglob('*'):
        if not folder.is_dir():
            continue
        if (folder / 'train').exists() or (folder / 'images').exists() or (folder / 'valid').exists() or (folder / 'test').exists():
            return folder

    return None

dataset_root = find_dataset_root()

if dataset_root is None:
    raise FileNotFoundError('Could not find data.yaml. Run the upload/extract and create data.yaml cells first.')

# Load YOLOv8 nano model (smallest and fastest for mobile)
model = YOLO('yolov8n.pt')

print('Starting YOLOv8 training with moderate augmentation...')
print('='*60)
print('Model: YOLOv8n (nano)')
print('Dataset:', dataset_root)
print('Classes: 5 (sigiriya_lion_paws, sigiriya_lion_rock, sigiriya_mirror_wall, sigiriya_throne, sigiriya_ticket_counter)')
print('='*60)

# Train the model with moderate augmentation
results = model.train(
    data=str(dataset_root / 'data.yaml'),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    project='/content/runs/detect',
    name='sigiriya_yolov8n',
    save=True,
    verbose=True,
    workers=4,

    # Moderate augmentation settings for heritage landmark detection
    augment=True,
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    translate=0.10,
    scale=0.50,
    shear=2.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.40,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=10
)

print('\n' + '='*60)
print('✓ Training complete!')
print('Best model: /content/runs/detect/sigiriya_yolov8n/weights/best.pt')
print('='*60)


## Step 6.5: Check for Overfitting
Compare training and validation curves after training finishes.

This helps you see whether the model is still improving on unseen data or starting to memorize the training set.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path('/content/runs/detect/sigiriya_yolov8n')
results_csv = results_dir / 'results.csv'

if not results_csv.exists():
    raise FileNotFoundError(f'Could not find {results_csv}. Run training first.')

# Load training history
metrics = pd.read_csv(results_csv)

# Ultralytics column names can vary slightly by version
train_box = metrics['train/box_loss'] if 'train/box_loss' in metrics.columns else None
val_box = metrics['val/box_loss'] if 'val/box_loss' in metrics.columns else None
train_cls = metrics['train/cls_loss'] if 'train/cls_loss' in metrics.columns else None
val_cls = metrics['val/cls_loss'] if 'val/cls_loss' in metrics.columns else None
map50 = metrics['metrics/mAP50(B)'] if 'metrics/mAP50(B)' in metrics.columns else None
map5095 = metrics['metrics/mAP50-95(B)'] if 'metrics/mAP50-95(B)' in metrics.columns else None
precision = metrics['metrics/precision(B)'] if 'metrics/precision(B)' in metrics.columns else None
recall = metrics['metrics/recall(B)'] if 'metrics/recall(B)' in metrics.columns else None

# Plot training vs validation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_box is not None and val_box is not None:
    axes[0].plot(train_box, label='train/box_loss')
    axes[0].plot(val_box, label='val/box_loss')
    axes[0].set_title('Box Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

if train_cls is not None and val_cls is not None:
    axes[1].plot(train_cls, label='train/cls_loss')
    axes[1].plot(val_cls, label='val/cls_loss')
    axes[1].set_title('Classification Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

plt.tight_layout()
plt.show()

# Simple overfitting heuristic using the last 10 epochs
last_n = min(10, len(metrics))
train_box_tail = train_box.tail(last_n).mean() if train_box is not None else None
val_box_tail = val_box.tail(last_n).mean() if val_box is not None else None
train_cls_tail = train_cls.tail(last_n).mean() if train_cls is not None else None
val_cls_tail = val_cls.tail(last_n).mean() if val_cls is not None else None

print('Final metrics summary:')
if precision is not None:
    print(f'  Precision: {precision.iloc[-1]:.3f}')
if recall is not None:
    print(f'  Recall:    {recall.iloc[-1]:.3f}')
if map50 is not None:
    print(f'  mAP50:     {map50.iloc[-1]:.3f}')
if map5095 is not None:
    print(f'  mAP50-95:  {map5095.iloc[-1]:.3f}')

print('\nOverfitting check:')
flags = []
if train_box_tail is not None and val_box_tail is not None:
    gap = val_box_tail - train_box_tail
    print(f'  Box loss gap (val - train): {gap:.3f}')
    if gap > 0.25:
        flags.append('box loss gap is large')
if train_cls_tail is not None and val_cls_tail is not None:
    gap = val_cls_tail - train_cls_tail
    print(f'  Cls loss gap (val - train): {gap:.3f}')
    if gap > 0.25:
        flags.append('cls loss gap is large')

if flags:
    print('  Possible overfitting signs: ' + ', '.join(flags))
else:
    print('  No strong overfitting signs detected from the final curves.')


## Step 7: Export to TensorFlow Lite
Convert the trained PyTorch model to TFLite format for mobile/Flutter deployment.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import os

def find_dataset_root():
    candidate_names = [
        'sigiriya_dataset',
        'sigiriya_dataset.yolov8',
        'sigiriya_dataset.v3-sigiriya_dataset_80-10-10.yolov8',
        'dataset',
    ]

    for name in candidate_names:
        direct = Path('/content') / name
        if (direct / 'train').exists() or (direct / 'images').exists() or (direct / 'data.yaml').exists():
            return direct

    for match in Path('/content').rglob('data.yaml'):
        parent = match.parent
        if (parent / 'train').exists() or (parent / 'images').exists() or (parent / 'valid').exists() or (parent / 'test').exists():
            return parent

    for folder in Path('/content').rglob('*'):
        if not folder.is_dir():
            continue
        if (folder / 'train').exists() or (folder / 'images').exists() or (folder / 'valid').exists() or (folder / 'test').exists():
            return folder

    return None

dataset_root = find_dataset_root()

if dataset_root is None:
    raise FileNotFoundError('Could not find data.yaml. Run the previous cells first.')

print('Exporting best model to TensorFlow Lite format...\n')

# Load the best trained model
best_model = YOLO('/content/runs/detect/sigiriya_yolov8n/weights/best.pt')
export_dir = Path('/content/runs/detect/sigiriya_yolov8n')

def find_latest_file(root_dir, suffix):
    matches = sorted(root_dir.rglob(f'*{suffix}'), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

try:
    # Export to TFLite (default: FP32, good for accuracy)
    export_result = best_model.export(format='tflite', imgsz=640)
    print('✓ TFLite export successful!')
    print(f'Exported model: {export_result}')
except Exception as e:
    print(f'Export error: {e}')
    print('Check that model training completed successfully')
    export_result = None

# Find the actual exported TFLite file, even if Ultralytics used a different subfolder
tflite_file = None
if isinstance(export_result, (str, Path)):
    candidate = Path(str(export_result))
    if candidate.exists() and candidate.suffix.lower() == '.tflite':
        tflite_file = candidate

if tflite_file is None:
    tflite_file = find_latest_file(export_dir, '.tflite')

# List all exported files
print('\n' + '='*60)
print('Exported files in /content/runs/detect/sigiriya_yolov8n/:')
print('='*60)
for path in sorted(export_dir.rglob('*')):
    if path.is_file() and path.suffix.lower() in {'.tflite', '.pt', '.onnx'}:
        size_mb = path.stat().st_size / (1024*1024)
        print(f'  ✓ {path.name} ({size_mb:.2f} MB) -> {path}')

if tflite_file is not None:
    print(f'\n✓ TFLite file located at: {tflite_file}')
else:
    print('\n⚠ No .tflite file found under the export directory yet.')


## Step 8: Save Models to Google Drive
Backup all trained models and metrics to your Google Drive.


In [ ]:
import shutil
from pathlib import Path

print("Saving models to Google Drive and preparing for download...\n")

# Create Drive output folder
drive_models = Path("/content/drive/MyDrive/heritage_ar_models")
drive_models.mkdir(parents=True, exist_ok=True)

def find_latest_file(root_dir, suffix):
    matches = sorted(root_dir.rglob(f'*{suffix}'), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

export_dir = Path('/content/runs/detect/sigiriya_yolov8n')

# Copy PyTorch checkpoint
pt_src = export_dir / 'weights' / 'best.pt'
if pt_src.exists():
    pt_dst = drive_models / 'sigiriya_best.pt'
    shutil.copy2(pt_src, pt_dst)
    print(f"✓ PyTorch model saved to Drive: {pt_dst}")

# Copy TFLite model
tflite_src = find_latest_file(export_dir, '.tflite')
if tflite_src is not None and tflite_src.exists():
    tflite_dst = drive_models / 'sigiriya_best.tflite'
    shutil.copy2(tflite_src, tflite_dst)
    print(f"✓ TFLite model saved to Drive: {tflite_dst}")
    print(f"   Size: {Path(tflite_dst).stat().st_size / (1024*1024):.2f} MB")
else:
    print("⚠ TFLite model not found in the export directory")
    print("  Run the export cell again and check its output above")

# Copy results metrics
results_src = export_dir / 'results.csv'
if results_src.exists():
    results_dst = drive_models / 'training_results.csv'
    shutil.copy2(results_src, results_dst)
    print(f"✓ Training metrics saved to Drive: {results_dst}")

print(f"\n✅ All files saved to: {drive_models}")


## Step 9: Download Models to Your Computer
Download the trained models for local use and Flutter integration.


In [ ]:
from google.colab import files
from pathlib import Path
import os

print("Preparing models for download to your computer...\n")

def find_latest_file(root_dir, suffix):
    matches = sorted(root_dir.rglob(f'*{suffix}'), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

export_dir = Path('/content/runs/detect/sigiriya_yolov8n')

# Download TFLite model (priority for Flutter)
tflite_path = find_latest_file(export_dir, '.tflite')
if tflite_path is not None and tflite_path.exists():
    print(f"📥 Downloading: {tflite_path.name}")
    files.download(str(tflite_path))
    print("✓ TFLite downloaded!")
else:
    print("⚠ TFLite not found")

# Download PyTorch checkpoint for future training
pt_path = export_dir / 'weights' / 'best.pt'
if pt_path.exists():
    print(f"\n📥 Downloading: best.pt (PyTorch checkpoint)")
    files.download(str(pt_path))
    print("✓ PyTorch model downloaded!")

print("\n" + "="*60)
print("✅ Download complete!")
print("="*60)
print("\nNext steps:")
print("1. Copy 'best.tflite' to: frontend/assets/models/landmark_model.tflite")
print("2. Keep 'best.pt' for future training/fine-tuning")
print("="*60)


## Summary

✅ **Training Complete!**

### What was trained:
- **Model:** YOLOv8n (nano - optimized for mobile)
- **Dataset:** 1,159 annotated images across 5 classes
- **Classes:**
  - sigiriya_lion_paws (337 images)
  - sigiriya_lion_rock (203 images)
  - sigiriya_mirror_wall (210 images)
  - sigiriya_throne (190 images)
  - sigiriya_ticket_counter (219 images)

### Downloaded Models:
- **best.tflite** - Mobile-ready model for Flutter AR
- **best.pt** - PyTorch checkpoint for future training

### Next Steps:
1. Copy `best.tflite` to `frontend/assets/models/landmark_model.tflite`
2. Update Flutter inference code to load the new model
3. Test on Android device with ARCore support

### Monitor Results:
- Check `/content/runs/detect/sigiriya_yolov8n/` for:
  - `results.csv` - Detailed training metrics
  - `plots/` - mAP, precision/recall curves
  - `confusion_matrix.png` - Class confusion matrix

---
**Project:** HeritageAR - Function 1: Sigiriya Sub-Landmark Detection  
**Date:** May 2026  
**Status:** ✅ Ready for Flutter Integration


In [ ]:
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# 1. Load the trained model
# Using a raw string (r'') prevents the invalid escape sequence warning.
# Please ensure the path points directly to the model file (.pt or .tflite).
model_path = r'E:\Campus website projects\final research\R26-IT-122\frontend\assets\models\landmark_model.tflite'

if not os.path.exists(model_path):
    print(f"❌ Model not found at {model_path}!")
else:
    model = YOLO(model_path)

    # 2. Select an image for local inference
    # Replace with the path to the image you want to test
    image_path = 'test_image.jpg' 

    if not os.path.exists(image_path):
        print(f"❌ Test image not found! Please place an image at: {image_path} or update the path.")
    else:
        # 3. Process the file
        img = Image.open(image_path).convert("RGB")

        # 4. Run Inference
        results = model.predict(source=img, conf=0.25)

        # 5. Display Result
        res_plotted = results[0].plot()
        plt.figure(figsize=(10, 8))
        plt.imshow(res_plotted)
        plt.axis('off')
        plt.title(f'Detection Result: {os.path.basename(image_path)}')
        plt.show()

        # 6. Print labels and confidence
        if len(results[0].boxes) == 0:
            print("No Sigiriya landmarks detected.")
        else:
            print("Results:")
            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                label = model.names[cls_id]
                print(f"  ✅ {label}: {conf:.2%} confidence")